# Laboratorio: números complejos y espacios complejos

Este laboratorio está separado de la hoja de ejercicios. Su propósito es comprobar y visualizar las estructuras estudiadas en la teoría. Al finalizar podrás:

- convertir entre las formas rectangular y polar;
- comprobar operaciones, potencias y raíces;
- visualizar las raíces $n$-ésimas;
- observar pares conjugados en polinomios reales;
- verificar productos internos y proyecciones en $\mathbb C^n$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

## 1. Forma rectangular y polar

NumPy escribe $i$ como `1j`. Para $z=a+bi$, `np.abs(z)` calcula el módulo y `np.angle(z)` devuelve un argumento principal.

In [ ]:
z = 1 - np.sqrt(3) * 1j
r = np.abs(z)
theta = np.angle(z)
z_reconstruido = r * np.exp(1j * theta)

print("z =", z)
print("módulo =", r)
print("argumento =", theta, "rad =", theta / np.pi, "π")
print("reconstrucción polar =", z_reconstruido)

assert np.allclose(z_reconstruido, z)

## 2. Operaciones y estructura de cuerpo

Comprobaremos el inverso multiplicativo y algunas identidades. Las tolerancias numéricas reemplazan la igualdad exacta porque se trabaja en punto flotante.

In [ ]:
z = 2 - 3j
w = 1 + 2j
inverso_z = np.conj(z) / np.abs(z)**2

print("z + w =", z + w)
print("z w =", z * w)
print("z / w =", z / w)
print("z^{-1} =", inverso_z)
print("z z^{-1} =", z * inverso_z)

assert np.allclose(z * inverso_z, 1)
assert np.allclose(np.conj(z * w), np.conj(z) * np.conj(w))
assert np.allclose(np.abs(z * w), np.abs(z) * np.abs(w))

## 3. Potencias y raíces $n$-ésimas

Para resolver $w^n=z=re^{i\theta}$ usamos

$$w_k=r^{1/n}e^{i(\theta+2k\pi)/n},\qquad k=0,\ldots,n-1.$$

In [ ]:
def raices_n_esimas(z, n):
    r = np.abs(z)
    theta = np.angle(z)
    k = np.arange(n)
    return r**(1/n) * np.exp(1j * (theta + 2 * np.pi * k) / n)

z = -8 + 0j
n = 3
raices = raices_n_esimas(z, n)
print("raíces =", raices)
print("potencias n-ésimas =", raices**n)

assert np.allclose(raices**n, z)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
circulo = np.linspace(0, 2 * np.pi, 400)
radio = np.abs(raices[0])
ax.plot(radio * np.cos(circulo), radio * np.sin(circulo), color="#9aa0a6", linewidth=1)
ax.scatter(raices.real, raices.imag, s=80, color="#be2d37", zorder=3)
for k, raiz in enumerate(raices):
    ax.plot([0, raiz.real], [0, raiz.imag], color="#14578c", linewidth=1)
    ax.annotate(f"$w_{k}$", (raiz.real, raiz.imag), xytext=(6, 6), textcoords="offset points")
ax.axhline(0, color="black", linewidth=0.7)
ax.axvline(0, color="black", linewidth=0.7)
ax.set_aspect("equal")
ax.set_xlabel("parte real")
ax.set_ylabel("parte imaginaria")
ax.set_title(r"Raíces de $w^3=-8$")
ax.grid(alpha=0.2)
plt.show()

## 4. Polinomios reales y pares conjugados

Las raíces no reales de un polinomio con coeficientes reales deben aparecer por pares conjugados.

In [ ]:
# p(t) = t^4 + 5t^2 + 4 = (t^2 + 1)(t^2 + 4)
coeficientes = [1, 0, 5, 0, 4]
raices_p = np.roots(coeficientes)
print("raíces de p =", raices_p)

for raiz in raices_p:
    assert np.min(np.abs(raices_p - np.conj(raiz))) < 1e-10

## 5. Producto interno y proyección en $\mathbb C^n$

Con la convención lineal en la primera entrada, `np.vdot(y, x)` calcula $y^*x=\langle x,y\rangle$.

In [ ]:
def producto_interno(x, y):
    return np.vdot(y, x)

u1 = np.array([1, 1j], dtype=complex) / np.sqrt(2)
u2 = np.array([1j, 1], dtype=complex) / np.sqrt(2)
x = np.array([2 - 1j, 1 + 3j], dtype=complex)

G = np.array([[producto_interno(u, v) for v in (u1, u2)] for u in (u1, u2)])
p = producto_interno(x, u1) * u1 + producto_interno(x, u2) * u2
residuo = x - p

print("matriz de productos internos =\n", G)
print("proyección =", p)
print("residuo =", residuo)

assert np.allclose(G, np.eye(2))
assert np.allclose(producto_interno(residuo, u1), 0)
assert np.allclose(producto_interno(residuo, u2), 0)

## 6. Actividades

1. Cambia `z` y `n` en la función `raices_n_esimas`. Comprueba que todas las raíces tienen el mismo módulo y argumentos igualmente espaciados.
2. Construye un polinomio real de grado seis con dos raíces reales y dos pares conjugados; comprueba sus raíces con `np.roots`.
3. Sustituye $u_2$ por $(i,-1)/\sqrt2$. ¿Qué revela la matriz de productos internos?
4. Elige un solo vector unitario $u$ y comprueba numéricamente la identidad $\|x\|^2=\|P_u x\|^2+\|x-P_u x\|^2$.